# Notebook 150: 埋め込みの幾何学 ― 距離・類似度・高次元の世界

## Geometry of Embeddings: Distance, Similarity, and High-Dimensional Worlds

---

### このノートブックの位置づけ

**Embeddings シリーズ** の第1章として、ベクトル空間に「意味」を埋め込む技術の幾何学的な基礎を学びます。

| 項目 | 内容 |
|------|------|
| 難易度 | ★★☆☆☆ |
| 所要時間 | 90〜120分 |
| カテゴリ | Embeddings |

### 学習目標

- [ ] ベクトル空間における「距離」と「類似度」の違いを説明できる
- [ ] コサイン類似度・ユークリッド距離・ドット積を実装し、使い分けられる
- [ ] 事前学習済み埋め込み（GloVe）を使って意味空間を探索できる
- [ ] 「高次元の呪い」が類似度計算に与える影響を体感できる
- [ ] 単語の演算（king - man + woman = queen）を幾何学的に理解できる

### 前提知識

- ✅ Notebook 70-76（ニューラルエンジン基礎）
- ✅ 線形代数の基礎（ベクトル、内積、ノルム）

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [なぜ「埋め込み」が必要か？](#2-なぜ埋め込みが必要か)
3. [距離と類似度の数理](#3-距離と類似度の数理)
4. [事前学習済み埋め込みの探索（GloVe）](#4-事前学習済み埋め込みの探索glove)
5. [単語の演算 — ベクトル算術の魔法](#5-単語の演算--ベクトル算術の魔法)
6. [高次元の呪い（Curse of Dimensionality）](#6-高次元の呪いcurse-of-dimensionality)
7. [距離指標の比較実験](#7-距離指標の比較実験)
8. [まとめ・チートシート・よくある間違い・自己評価クイズ](#8-まとめチートシートよくある間違い自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 日本語フォント設定（環境に応じて調整）
# ------------------------------------------------------------
def setup_japanese_font():
    """日本語フォントをmatplotlibに設定する関数"""
    # macOS: Hiragino Sans, Windows: MS Gothic, Linux: IPAGothic
    font_candidates = ['Hiragino Sans', 'Arial Unicode MS', 'IPAGothic', 'MS Gothic', 'sans-serif']
    plt.rcParams['font.family'] = font_candidates
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (10, 6)
    plt.rcParams['font.size'] = 11
    print("✅ 日本語フォント設定完了")

setup_japanese_font()

# ------------------------------------------------------------
# 再現性のためのシード
# ------------------------------------------------------------
np.random.seed(42)

# Seaborn のスタイル設定
sns.set_style('whitegrid')

print("✅ 環境セットアップ完了")
print(f"   NumPy version: {np.__version__}")

---

## 2. なぜ「埋め込み」が必要か？

### 2.1 One-Hot Encoding の限界

自然言語をコンピュータで扱う最も素朴な方法は **One-Hot Encoding** です。語彙数 $V$ のとき、各単語を $V$ 次元のベクトルで表します。

```
語彙: [cat, dog, king, queen, man, woman]  (V=6)

One-Hot 表現:
  cat   = [1, 0, 0, 0, 0, 0]
  dog   = [0, 1, 0, 0, 0, 0]
  king  = [0, 0, 1, 0, 0, 0]
  queen = [0, 0, 0, 1, 0, 0]
  man   = [0, 0, 0, 0, 1, 0]
  woman = [0, 0, 0, 0, 0, 1]
```

**問題点：**

1. **直交性**: すべてのベクトルが互いに直交 → `cat` と `dog` の類似度 = `cat` と `king` の類似度 = 0
2. **高次元・疎**: 語彙が10万語なら10万次元、ほとんどが0
3. **意味の欠如**: ベクトルに「意味」の情報が一切含まれない

### 2.2 密なベクトル表現（Dense Embedding）

```
Dense Embedding (d=3 の例):
  cat   = [0.2,  0.8, -0.1]    ← 「動物」方向に大きい
  dog   = [0.3,  0.7,  0.0]    ← cat に近い！
  king  = [-0.5, 0.1,  0.9]    ← 「王族」方向に大きい
  queen = [-0.4, 0.0,  0.8]    ← king に近い！
```

**利点：**
- 低次元（50〜300次元）で意味を表現
- 類似した単語が近くに配置される
- ベクトル演算で「意味」を操作できる

In [ ]:
# ============================================================
# Section 2: One-Hot vs Dense Embedding の比較
# ============================================================

# --- One-Hot Encoding ---
words = ['cat', 'dog', 'king', 'queen', 'man', 'woman']
V = len(words)

# 各単語の One-Hot ベクトルを作成
onehot_vectors = np.eye(V)

print("=" * 60)
print("【One-Hot Encoding】")
print("=" * 60)
for i, word in enumerate(words):
    print(f"  {word:>6} = {onehot_vectors[i]}")

# One-Hot ベクトル間のドット積（内積）
print("\n--- One-Hot ベクトル間のドット積 ---")
pairs = [('cat', 'dog'), ('cat', 'king'), ('king', 'queen')]
for w1, w2 in pairs:
    i1, i2 = words.index(w1), words.index(w2)
    dot = np.dot(onehot_vectors[i1], onehot_vectors[i2])
    print(f"  {w1:>6} · {w2:<6} = {dot:.1f}  ← すべて同じ！意味の近さが反映されない")

print()

# --- 仮想的な Dense Embedding ---
# 3次元の密なベクトル（手動で意味が反映されるように設定）
dense_vectors = {
    'cat':   np.array([ 0.2,  0.8, -0.1]),
    'dog':   np.array([ 0.3,  0.7,  0.0]),
    'king':  np.array([-0.5,  0.1,  0.9]),
    'queen': np.array([-0.4,  0.0,  0.8]),
    'man':   np.array([-0.3,  0.2,  0.5]),
    'woman': np.array([-0.2,  0.1,  0.4]),
}

print("=" * 60)
print("【Dense Embedding (3次元)】")
print("=" * 60)
for word in words:
    print(f"  {word:>6} = {dense_vectors[word]}")

# Dense ベクトル間のドット積
print("\n--- Dense ベクトル間のドット積 ---")
for w1, w2 in pairs:
    dot = np.dot(dense_vectors[w1], dense_vectors[w2])
    print(f"  {w1:>6} · {w2:<6} = {dot:.3f}")

print("\n✅ Dense Embedding では、意味的に近い単語ほどドット積が大きい！")

In [ ]:
# ============================================================
# One-Hot vs Dense の類似度ヒートマップ比較
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左: One-Hot のドット積行列 ---
onehot_sim = onehot_vectors @ onehot_vectors.T
sns.heatmap(onehot_sim, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=words, yticklabels=words, ax=axes[0],
            vmin=0, vmax=1, square=True)
axes[0].set_title('One-Hot: ドット積行列\n（自分自身以外すべて0）', fontsize=13)

# --- 右: Dense Embedding のコサイン類似度行列 ---
dense_matrix = np.array([dense_vectors[w] for w in words])
# コサイン類似度を計算
norms = np.linalg.norm(dense_matrix, axis=1, keepdims=True)
dense_normed = dense_matrix / norms
dense_cos_sim = dense_normed @ dense_normed.T

sns.heatmap(dense_cos_sim, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=words, yticklabels=words, ax=axes[1],
            vmin=-1, vmax=1, square=True)
axes[1].set_title('Dense Embedding: コサイン類似度行列\n（意味の近さが数値に反映）', fontsize=13)

plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・One-Hot: 対角要素のみ1、他はすべて0（情報量ゼロ）")
print("・Dense:   cat-dog が高い類似度、king-queen も高い類似度")
print("  → 意味的関係がベクトルの幾何学に反映されている")

---

## 3. 距離と類似度の数理

### 3.1 三つの基本指標

ベクトル $\mathbf{a}$ と $\mathbf{b}$ の「近さ」を測る代表的な指標は3つあります。

| 指標 | 数式 | 性質 |
|------|------|------|
| ユークリッド距離 | $d(\mathbf{a}, \mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|_2 = \sqrt{\sum_i (a_i - b_i)^2}$ | 小さいほど近い |
| コサイン類似度 | $\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$ | 大きいほど近い（方向のみ）|
| ドット積 | $\mathbf{a} \cdot \mathbf{b} = \sum_i a_i b_i$ | 方向 + 大きさ |

### 3.2 「距離」と「類似度」の違い

- **距離（Distance）**: 小さいほど「近い」→ 0 が最も近い
- **類似度（Similarity）**: 大きいほど「近い」→ 1 が最も近い（コサイン類似度の場合）

この区別は検索やランキングで非常に重要です。

In [ ]:
# ============================================================
# Section 3: 距離・類似度の実装（NumPy でゼロから）
# ============================================================

def euclidean_distance(a, b):
    """
    ユークリッド距離を計算する。
    d(a, b) = ||a - b||₂ = sqrt(Σ(aᵢ - bᵢ)²)
    
    Parameters:
        a, b: numpy配列（同じ次元）
    Returns:
        float: ユークリッド距離（非負）
    """
    diff = a - b
    return np.sqrt(np.sum(diff ** 2))


def cosine_similarity_manual(a, b):
    """
    コサイン類似度を計算する。
    cos(a, b) = (a · b) / (||a|| ||b||)
    
    Parameters:
        a, b: numpy配列（同じ次元）
    Returns:
        float: コサイン類似度（-1 から 1）
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    # ゼロベクトルのチェック
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)


def dot_product(a, b):
    """
    ドット積（内積）を計算する。
    a · b = Σ aᵢbᵢ
    
    Parameters:
        a, b: numpy配列（同じ次元）
    Returns:
        float: ドット積
    """
    return np.sum(a * b)


# --- テスト ---
print("=" * 60)
print("距離・類似度の実装テスト")
print("=" * 60)

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
c = np.array([-1.0, -2.0, -3.0])  # a の逆方向

print(f"\nベクトル a = {a}")
print(f"ベクトル b = {b}")
print(f"ベクトル c = {c}  (a の逆方向)")

print(f"\n--- a と b の比較 ---")
print(f"  ユークリッド距離: {euclidean_distance(a, b):.4f}")
print(f"  コサイン類似度:   {cosine_similarity_manual(a, b):.4f}")
print(f"  ドット積:         {dot_product(a, b):.4f}")

print(f"\n--- a と c の比較 ---")
print(f"  ユークリッド距離: {euclidean_distance(a, c):.4f}")
print(f"  コサイン類似度:   {cosine_similarity_manual(a, c):.4f}  ← 逆方向なので -1")
print(f"  ドット積:         {dot_product(a, c):.4f}")

# --- NumPy の組み込み関数との検証 ---
print(f"\n✅ NumPy検証: ユークリッド距離 = {np.linalg.norm(a - b):.4f}")
print(f"✅ NumPy検証: ドット積 = {np.dot(a, b):.4f}")

In [ ]:
# ============================================================
# 2Dベクトルで角度 vs 距離を可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# テスト用の2Dベクトル群
vectors_2d = {
    'a': np.array([3.0, 1.0]),
    'b': np.array([1.0, 3.0]),
    'c': np.array([2.0, 2.0]),
    'd': np.array([0.5, 0.5]),   # c と同じ方向、短い
    'e': np.array([-2.0, 1.0]),  # 異なる方向
}

colors = {'a': '#e74c3c', 'b': '#3498db', 'c': '#2ecc71', 'd': '#f39c12', 'e': '#9b59b6'}

# --- 左: ベクトルの可視化 ---
ax = axes[0]
for name, vec in vectors_2d.items():
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors[name], lw=2.5))
    offset = vec / np.linalg.norm(vec) * 0.3
    ax.text(vec[0] + offset[0], vec[1] + offset[1], f'{name}',
            fontsize=14, fontweight='bold', color=colors[name])
ax.set_xlim(-3, 4)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('5つの2Dベクトル', fontsize=13)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.grid(True, alpha=0.3)

# --- 中央: コサイン類似度の比較 ---
ax = axes[1]
ref = 'a'  # 基準ベクトル
other_names = ['b', 'c', 'd', 'e']
cos_sims = [cosine_similarity_manual(vectors_2d[ref], vectors_2d[n]) for n in other_names]
bar_colors = [colors[n] for n in other_names]
bars = ax.barh(other_names, cos_sims, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('コサイン類似度', fontsize=12)
ax.set_title(f'ベクトル a との\nコサイン類似度', fontsize=13)
ax.set_xlim(-1, 1)
ax.axvline(x=0, color='k', linewidth=0.5)
# 値をバーの上に表示
for bar, val in zip(bars, cos_sims):
    ax.text(val + 0.05 if val >= 0 else val - 0.15, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

# --- 右: ユークリッド距離の比較 ---
ax = axes[2]
euc_dists = [euclidean_distance(vectors_2d[ref], vectors_2d[n]) for n in other_names]
bars = ax.barh(other_names, euc_dists, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('ユークリッド距離', fontsize=12)
ax.set_title(f'ベクトル a との\nユークリッド距離', fontsize=13)
# 値をバーの上に表示
for bar, val in zip(bars, euc_dists):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("【重要な観察】")
print("・c と d は同じ方向 → コサイン類似度は同じ（1.0）")
print("  しかしユークリッド距離は異なる（ベクトルの長さが違う）")
print("・コサイン類似度は『方向』だけ、ユークリッド距離は『位置』を見る")
print("・用途に応じて使い分けが重要！")

In [ ]:
# ============================================================
# 3つの指標の使い分け: 比較表
# ============================================================

print("=" * 75)
print("距離・類似度指標の使い分けガイド")
print("=" * 75)
print()
print(f"{'指標':<18} {'値域':<14} {'大きさ感度':<12} {'主な用途'}")
print("-" * 75)
print(f"{'ユークリッド距離':<16} {'[0, ∞)':<14} {'あり':<12} {'クラスタリング、kNN'}")
print(f"{'コサイン類似度':<16}  {'[-1, 1]':<14} {'なし（方向のみ）':<12} {'文書検索、意味類似度'}")
print(f"{'ドット積':<16}     {'(-∞, ∞)':<14} {'あり':<12} {'Attention、推薦'}")
print()
print("【いつどれを使うか？】")
print("  📌 正規化済みベクトル   → コサイン類似度 ≒ ドット積（等価）")
print("  📌 ベクトルの大きさが重要 → ユークリッド距離 or ドット積")
print("  📌 方向だけ比較したい    → コサイン類似度")
print("  📌 Transformerの中       → スケールドドット積")
print()
print("💡 ヒント: 正規化済みベクトル（||v||=1）では以下が成り立つ:")
print("   cos(a,b) = a · b")
print("   ||a - b||² = 2(1 - cos(a,b))")
print("   → 3つの指標が数学的に等価になる！")

---

## 4. 事前学習済み埋め込みの探索（GloVe）

### 4.1 GloVe とは

**GloVe (Global Vectors for Word Representation)** は、Stanford大学が開発した単語埋め込みです。大規模テキストコーパスから、単語の共起統計量に基づいて学習されています。

ここでは `gensim` ライブラリを使って、50次元の GloVe ベクトルをロードします。

In [ ]:
# ============================================================
# Section 4: GloVe 事前学習済み埋め込みの読み込み
# ============================================================
import gensim.downloader as api

# GloVe 50次元ベクトルをダウンロード・ロード
# （初回はダウンロードに数分かかります）
print("⏳ GloVe ベクトルをロード中...")
glove = api.load('glove-wiki-gigaword-50')
print(f"✅ ロード完了！")
print(f"   語彙数:   {len(glove):,} 語")
print(f"   次元数:   {glove.vector_size}")
print(f"   例: 'king' のベクトル shape = {glove['king'].shape}")

In [ ]:
# ============================================================
# 類似単語の探索
# ============================================================

# いくつかの単語について、最も類似した単語を検索
query_words = ['king', 'computer', 'japan']

print("=" * 60)
print("GloVe による類似単語検索（コサイン類似度 Top-10）")
print("=" * 60)

for word in query_words:
    print(f"\n🔍 '{word}' に最も類似した単語:")
    print(f"   {'順位':<4} {'単語':<15} {'コサイン類似度'}")
    print(f"   {'-'*40}")
    similar = glove.most_similar(word, topn=10)
    for rank, (sim_word, score) in enumerate(similar, 1):
        # 類似度の高さをバーで可視化
        bar = '█' * int(score * 20)
        print(f"   {rank:<4} {sim_word:<15} {score:.4f}  {bar}")

In [ ]:
# ============================================================
# 単語間の類似度ヒートマップ
# ============================================================

# 意味的にグループを作れる単語を選ぶ
selected_words = [
    # 王族
    'king', 'queen', 'prince', 'princess',
    # 国
    'japan', 'china', 'france', 'germany',
    # 動物
    'cat', 'dog', 'fish', 'bird',
    # テクノロジー
    'computer', 'software', 'internet', 'phone',
]

# 各単語のベクトルを取得
word_vectors = np.array([glove[w] for w in selected_words])

# コサイン類似度行列を計算
sim_matrix = cosine_similarity(word_vectors)

# ヒートマップ
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='RdYlBu_r',
            xticklabels=selected_words, yticklabels=selected_words,
            ax=ax, vmin=-0.5, vmax=1.0, square=True,
            linewidths=0.5, linecolor='white')
ax.set_title('GloVe 50d: 単語間コサイン類似度ヒートマップ', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・同じカテゴリの単語同士（king-queen, japan-china等）は高い類似度")
print("・異なるカテゴリ間の類似度は相対的に低い")
print("・GloVe は大規模テキストの共起情報だけで、これだけの意味構造を獲得している")

In [ ]:
# ============================================================
# 単語クラスタの可視化（PCA で 2D に射影）
# ============================================================

# PCA で 50次元 → 2次元に次元削減
pca = PCA(n_components=2)
vectors_2d_pca = pca.fit_transform(word_vectors)

# 色分け用のカテゴリ
categories = (['王族'] * 4 + ['国'] * 4 + ['動物'] * 4 + ['テクノロジー'] * 4)
category_colors = {
    '王族': '#e74c3c',
    '国': '#3498db',
    '動物': '#2ecc71',
    'テクノロジー': '#9b59b6'
}

fig, ax = plt.subplots(figsize=(12, 8))

for i, (word, cat) in enumerate(zip(selected_words, categories)):
    x, y = vectors_2d_pca[i]
    color = category_colors[cat]
    ax.scatter(x, y, c=color, s=120, zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(word, (x, y), textcoords='offset points', xytext=(8, 5),
                fontsize=11, fontweight='bold', color=color)

# 凡例
legend_patches = [mpatches.Patch(color=c, label=cat) for cat, c in category_colors.items()]
ax.legend(handles=legend_patches, fontsize=11, loc='upper left')

ax.set_xlabel(f'第1主成分（寄与率: {pca.explained_variance_ratio_[0]:.1%}）', fontsize=12)
ax.set_ylabel(f'第2主成分（寄与率: {pca.explained_variance_ratio_[1]:.1%}）', fontsize=12)
ax.set_title('GloVe 単語ベクトルの PCA 射影（50D → 2D）', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ 同じカテゴリの単語が2D空間上でもクラスタを形成している")
print(f"   注意: 2次元への射影で失われた情報 = {1 - sum(pca.explained_variance_ratio_):.1%}")

---

## 5. 単語の演算 — ベクトル算術の魔法

### 5.1 有名な例: king - man + woman = ?

埋め込み空間の最も驚くべき性質の一つは、**ベクトルの演算が意味の演算に対応する**ことです。

$$
\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}
$$

幾何学的な解釈：
- `king - man` = 「男性」成分を取り除いた「王族」ベクトル
- `+ woman` = 「女性」成分を加える
- 結果 ≈ 「女性の王族」= queen

In [ ]:
# ============================================================
# Section 5: 単語ベクトルの演算
# ============================================================

def word_analogy(model, positive, negative, topn=5):
    """
    単語の演算（アナロジー）を実行する。
    
    positive に含まれる単語のベクトルを足し、
    negative に含まれる単語のベクトルを引いた結果に
    最も近い単語を返す。
    
    Parameters:
        model: gensim の KeyedVectors
        positive: 足す単語のリスト
        negative: 引く単語のリスト
        topn: 上位何件返すか
    Returns:
        list of (word, similarity)
    """
    result = model.most_similar(positive=positive, negative=negative, topn=topn)
    return result


# --- 有名なアナロジーテスト ---
analogies = [
    # (positive, negative, 期待される答え, 説明)
    (['king', 'woman'], ['man'], 'queen', 'king - man + woman = ?'),
    (['japan', 'paris'], ['tokyo'], 'france', 'japan - tokyo + paris = ?'),
    (['walked', 'swimming'], ['walking'], 'swam', 'walked - walking + swimming = ?'),
    (['bigger', 'cold'], ['big'], 'colder', 'bigger - big + cold = ?'),
]

print("=" * 70)
print("単語ベクトルの演算（Analogy Test）")
print("=" * 70)

for positive, negative, expected, description in analogies:
    results = word_analogy(glove, positive, negative, topn=5)
    top_word = results[0][0]
    # 期待通りかチェック
    match = '✅' if top_word == expected else '⚠️'
    
    print(f"\n{match} {description}")
    print(f"   式: {' + '.join(positive)} - {' - '.join(negative)}")
    print(f"   期待: {expected}")
    print(f"   結果 Top-5:")
    for rank, (word, score) in enumerate(results, 1):
        marker = ' ★' if word == expected else ''
        print(f"     {rank}. {word:<15} ({score:.4f}){marker}")

In [ ]:
# ============================================================
# アナロジーの幾何学的可視化（平行四辺形）
# ============================================================

# king - man + woman ≈ queen を2D PCA で可視化
analogy_words = ['king', 'queen', 'man', 'woman']
analogy_vectors = np.array([glove[w] for w in analogy_words])

# PCA で 2D に射影
pca_analogy = PCA(n_components=2)
analogy_2d = pca_analogy.fit_transform(analogy_vectors)

fig, ax = plt.subplots(figsize=(10, 8))

# 各単語をプロット
colors_analogy = ['#e74c3c', '#e74c3c', '#3498db', '#3498db']
markers_analogy = ['s', 'D', 's', 'D']  # 四角=男性系, ダイヤ=女性系

for i, (word, color, marker) in enumerate(zip(analogy_words, colors_analogy, markers_analogy)):
    ax.scatter(analogy_2d[i, 0], analogy_2d[i, 1], c=color, s=200,
              marker=marker, zorder=5, edgecolors='black', linewidth=1)
    ax.annotate(word, analogy_2d[i], textcoords='offset points',
                xytext=(12, 12), fontsize=16, fontweight='bold', color=color)

# 「性別」方向の矢印（man → woman, king → queen）
ax.annotate('', xy=analogy_2d[3], xytext=analogy_2d[2],  # man → woman
            arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2.5, linestyle='--'))
ax.annotate('', xy=analogy_2d[1], xytext=analogy_2d[0],  # king → queen
            arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2.5, linestyle='--'))

# 「王族」方向の矢印（man → king, woman → queen）
ax.annotate('', xy=analogy_2d[0], xytext=analogy_2d[2],  # man → king
            arrowprops=dict(arrowstyle='->', color='#f39c12', lw=2.5, linestyle=':'))
ax.annotate('', xy=analogy_2d[1], xytext=analogy_2d[3],  # woman → queen
            arrowprops=dict(arrowstyle='->', color='#f39c12', lw=2.5, linestyle=':'))

# 凡例用のダミー
ax.plot([], [], color='#2ecc71', linestyle='--', linewidth=2.5, label='性別方向 (man→woman)')
ax.plot([], [], color='#f39c12', linestyle=':', linewidth=2.5, label='王族方向 (man→king)')

ax.set_xlabel('第1主成分', fontsize=12)
ax.set_ylabel('第2主成分', fontsize=12)
ax.set_title('単語アナロジーの幾何学: king - man + woman ≈ queen\n（平行四辺形構造）', fontsize=14)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【幾何学的解釈】")
print("・king→queen と man→woman のベクトルがほぼ平行")
print("  → 『性別』という意味の軸が埋め込み空間に存在する")
print("・man→king と woman→queen のベクトルもほぼ平行")
print("  → 『王族』という意味の軸も存在する")
print("・4つの単語が平行四辺形を形成 = アナロジーが成立する幾何学的条件")

In [ ]:
# ============================================================
# 複数のアナロジー例を表にまとめる
# ============================================================

# 追加のアナロジーテスト
extended_analogies = [
    ('king', 'man', 'woman', 'queen',     '性別'),
    ('japan', 'tokyo', 'paris', 'france',  '国-首都'),
    ('good', 'better', 'big', 'bigger',    '比較級'),
    ('slow', 'slowly', 'quick', 'quickly', '副詞化'),
    ('man', 'men', 'woman', 'women',       '複数形'),
]

print("=" * 75)
print("アナロジーテスト結果一覧")
print("=" * 75)
print(f"  {'関係':<10} {'A':<10} {'B':<10} {'C':<10} {'期待D':<10} {'実際Top1':<10} {'結果'}")
print(f"  {'-'*70}")
print(f"  {'':>10} A - B + C ≈ D")
print(f"  {'-'*70}")

for a, b, c, expected_d, relation in extended_analogies:
    try:
        results = glove.most_similar(positive=[a, c], negative=[b], topn=3)
        top1 = results[0][0]
        match = '✅' if top1 == expected_d else '⚠️'
        print(f"  {relation:<10} {a:<10} {b:<10} {c:<10} {expected_d:<10} {top1:<10} {match}")
    except KeyError as e:
        print(f"  {relation:<10} {a:<10} {b:<10} {c:<10} {expected_d:<10} {'(語彙外)':<10} ❌")

print("\n💡 50次元の GloVe でもこれだけの言語関係を捉えている")
print("   （より高次元の 300d モデルならさらに精度が向上）")

---

## 6. 高次元の呪い（Curse of Dimensionality）

### 6.1 なぜ高次元は「呪い」なのか

直感に反する高次元空間の性質が、類似度計算や最近傍探索に深刻な影響を与えます。

**主な現象：**
1. ランダムなベクトル同士のコサイン類似度が **0 付近に集中** する
2. 最近傍と最遠傍の **距離の比が 1 に近づく**（全員が同じくらい遠い）
3. 体積のほとんどが「表面」付近に集中する

これらの現象を実験で確認しましょう。

In [ ]:
# ============================================================
# Section 6: 高次元の呪い — 実験
# ============================================================

# 実験: 異なる次元でランダム単位ベクトルを生成し、
# ペアワイズのコサイン類似度の分布を調べる

dimensions = [2, 10, 50, 100, 500, 1000]
n_vectors = 500  # 各次元で生成するベクトル数

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

# 結果を保存
stats_list = []

for idx, dim in enumerate(dimensions):
    # ランダムな単位ベクトルを生成
    # （正規分布からサンプリングして正規化するのが一般的な方法）
    vectors = np.random.randn(n_vectors, dim)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    unit_vectors = vectors / norms
    
    # ペアワイズのコサイン類似度を計算
    cos_sim_matrix = unit_vectors @ unit_vectors.T
    
    # 上三角部分（自分自身を除く）のみ抽出
    upper_tri_indices = np.triu_indices(n_vectors, k=1)
    similarities = cos_sim_matrix[upper_tri_indices]
    
    # 統計量を記録
    stats_list.append({
        'dim': dim,
        'mean': np.mean(similarities),
        'std': np.std(similarities),
        'min': np.min(similarities),
        'max': np.max(similarities),
    })
    
    # ヒストグラム
    ax = axes[idx]
    ax.hist(similarities, bins=50, density=True, alpha=0.7,
            color=plt.cm.viridis(idx / len(dimensions)), edgecolor='black', linewidth=0.3)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='cos=0')
    ax.set_xlabel('コサイン類似度', fontsize=10)
    ax.set_ylabel('密度', fontsize=10)
    ax.set_title(f'{dim}次元 (std={np.std(similarities):.4f})', fontsize=12)
    ax.set_xlim(-1, 1)
    ax.legend(fontsize=9)

plt.suptitle('高次元の呪い: ランダム単位ベクトルのコサイン類似度分布', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 統計量の表示
print("\n" + "=" * 65)
print("コサイン類似度の統計量（ランダム単位ベクトル間）")
print("=" * 65)
print(f"  {'次元':>6} {'平均':>10} {'標準偏差':>10} {'最小':>10} {'最大':>10}")
print(f"  {'-'*50}")
for s in stats_list:
    print(f"  {s['dim']:>6} {s['mean']:>10.4f} {s['std']:>10.4f} {s['min']:>10.4f} {s['max']:>10.4f}")

print("\n⚠️ 次元が上がるにつれ、全ペアの類似度が 0 付近に集中！")
print("   → 高次元では『ランダムなベクトルはほぼ直交する』")

In [ ]:
# ============================================================
# 距離の比（最近傍/最遠傍）の収束実験
# ============================================================

dimensions_exp = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
n_points = 200  # データ点の数
n_queries = 50  # クエリ点の数

distance_ratios = []  # 各次元での距離比

for dim in dimensions_exp:
    # ランダムな点を生成
    data_points = np.random.randn(n_points, dim)
    query_points = np.random.randn(n_queries, dim)
    
    ratios = []
    for q in query_points:
        # クエリ点から全データ点へのユークリッド距離
        distances = np.linalg.norm(data_points - q, axis=1)
        # 最近傍距離 / 最遠傍距離
        ratio = np.min(distances) / np.max(distances)
        ratios.append(ratio)
    
    distance_ratios.append({
        'dim': dim,
        'mean_ratio': np.mean(ratios),
        'std_ratio': np.std(ratios),
    })

# 可視化
fig, ax = plt.subplots(figsize=(10, 6))

dims = [r['dim'] for r in distance_ratios]
means = [r['mean_ratio'] for r in distance_ratios]
stds = [r['std_ratio'] for r in distance_ratios]

ax.errorbar(dims, means, yerr=stds, fmt='o-', linewidth=2, markersize=8,
            capsize=5, color='#e74c3c', ecolor='gray')
ax.axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='比 = 1.0（完全に区別不能）')

ax.set_xlabel('次元数', fontsize=12)
ax.set_ylabel('最近傍距離 / 最遠傍距離', fontsize=12)
ax.set_title('高次元の呪い: 距離の意味が薄れる\n（比が1に近づくほど、最近傍と最遠傍の区別がつかない）', fontsize=13)
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("【距離比の収束】")
print(f"  2次元:    最近傍/最遠傍 = {distance_ratios[0]['mean_ratio']:.3f}  ← 区別できる")
print(f"  1000次元: 最近傍/最遠傍 = {distance_ratios[-1]['mean_ratio']:.3f}  ← ほぼ同じ距離")
print("\n⚠️ 高次元では kNN が機能しにくくなる一因")
print("💡 これが、次元削減や局所的ハッシュ（LSH）が重要な理由")

---

## 7. 距離指標の比較実験

### 7.1 GloVe ベクトルでの指標比較

実際の GloVe 埋め込みを使って、3つの距離指標で「最も似ている単語」のランキングがどう変わるかを比較します。

In [ ]:
# ============================================================
# Section 7: GloVe ベクトルでの距離指標の比較
# ============================================================

def rank_by_metric(query_word, candidate_words, model, metric='cosine'):
    """
    指定された距離指標でcandidate_wordsをランク付けする。
    
    Parameters:
        query_word: クエリ単語
        candidate_words: 候補単語のリスト
        model: gensim KeyedVectors
        metric: 'cosine', 'euclidean', 'dot'
    Returns:
        list of (word, score) sorted by relevance (descending)
    """
    query_vec = model[query_word]
    scores = []
    
    for word in candidate_words:
        vec = model[word]
        if metric == 'cosine':
            # コサイン類似度（大きいほど近い）
            score = cosine_similarity_manual(query_vec, vec)
        elif metric == 'euclidean':
            # ユークリッド距離（小さいほど近い → 負にして大きいほど近いに統一）
            score = -euclidean_distance(query_vec, vec)
        elif metric == 'dot':
            # ドット積（大きいほど近い）
            score = dot_product(query_vec, vec)
        scores.append((word, score))
    
    # スコアの降順でソート
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores


# テスト用の単語セット
query = 'king'
candidates = ['queen', 'prince', 'princess', 'man', 'woman', 'throne',
              'royal', 'kingdom', 'castle', 'crown',
              'dog', 'cat', 'car', 'computer', 'water']

metrics = ['cosine', 'euclidean', 'dot']
metric_names = {'cosine': 'コサイン類似度', 'euclidean': 'ユークリッド距離', 'dot': 'ドット積'}

# 各指標でランキング
rankings = {}
for metric in metrics:
    rankings[metric] = rank_by_metric(query, candidates, glove, metric)

# 結果表示
print(f"クエリ: '{query}' に対する候補単語のランキング比較")
print("=" * 75)
print(f"{'順位':<4}", end='')
for m in metrics:
    print(f"  {metric_names[m]:<22}", end='')
print()
print("-" * 75)

for i in range(len(candidates)):
    print(f"{i+1:<4}", end='')
    for m in metrics:
        word, score = rankings[m][i]
        print(f"  {word:<15} {score:>6.3f}", end='')
    print()

In [ ]:
# ============================================================
# コサイン類似度 vs ユークリッド距離のランキング比較
# ============================================================

# 各指標でのランク（1-indexed）を取得
def get_rank_dict(ranked_list):
    """ランキングリストから {word: rank} の辞書を返す"""
    return {word: rank + 1 for rank, (word, _) in enumerate(ranked_list)}

cosine_ranks = get_rank_dict(rankings['cosine'])
euclidean_ranks = get_rank_dict(rankings['euclidean'])
dot_ranks = get_rank_dict(rankings['dot'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- 左: コサイン vs ユークリッド ---
ax = axes[0]
for word in candidates:
    cr = cosine_ranks[word]
    er = euclidean_ranks[word]
    color = '#e74c3c' if abs(cr - er) >= 3 else '#3498db'
    ax.scatter(cr, er, c=color, s=80, zorder=5)
    ax.annotate(word, (cr, er), textcoords='offset points',
                xytext=(5, 5), fontsize=9)

# 完全一致の対角線
ax.plot([0, len(candidates)+1], [0, len(candidates)+1], 'k--', alpha=0.3, label='完全一致')
ax.set_xlabel('コサイン類似度のランク', fontsize=12)
ax.set_ylabel('ユークリッド距離のランク', fontsize=12)
ax.set_title(f"'{query}' に対するランキング比較\nコサイン vs ユークリッド", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, len(candidates)+1)
ax.set_ylim(0, len(candidates)+1)
ax.set_aspect('equal')

# --- 右: コサイン vs ドット積 ---
ax = axes[1]
for word in candidates:
    cr = cosine_ranks[word]
    dr = dot_ranks[word]
    color = '#e74c3c' if abs(cr - dr) >= 3 else '#3498db'
    ax.scatter(cr, dr, c=color, s=80, zorder=5)
    ax.annotate(word, (cr, dr), textcoords='offset points',
                xytext=(5, 5), fontsize=9)

ax.plot([0, len(candidates)+1], [0, len(candidates)+1], 'k--', alpha=0.3, label='完全一致')
ax.set_xlabel('コサイン類似度のランク', fontsize=12)
ax.set_ylabel('ドット積のランク', fontsize=12)
ax.set_title(f"'{query}' に対するランキング比較\nコサイン vs ドット積", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, len(candidates)+1)
ax.set_ylim(0, len(candidates)+1)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("【観察ポイント】")
print("・対角線上の点: 両指標でランクが一致する単語")
print("・対角線から外れる点（赤色）: 指標によってランクが変わる単語")
print("・コサインとユークリッドはかなり似た順序を返すことが多い")
print("・ドット積はベクトルの大きさの影響を受けるため、順序が異なりやすい")

---

## 8. まとめ・チートシート・よくある間違い・自己評価クイズ

### 8.1 まとめ

このノートブックでは、埋め込み（Embedding）の幾何学的な基礎を学びました。

1. **One-Hot vs Dense Embedding**: 疎で意味のないベクトルから、密で意味を持つベクトルへ
2. **距離と類似度**: ユークリッド距離（位置）、コサイン類似度（方向）、ドット積（方向+大きさ）
3. **GloVe 探索**: 事前学習済み埋め込みが意味的クラスタを形成していることを確認
4. **単語の演算**: ベクトル算術が意味の算術に対応する（平行四辺形構造）
5. **高次元の呪い**: 次元が高くなると類似度が集中し、距離の区別がつかなくなる
6. **指標の使い分け**: 用途に応じて適切な距離・類似度指標を選ぶことが重要

### 8.2 チートシート

| 距離指標 | 数式 | 特徴 | 使い分け |
|---------|------|------|----------|
| ユークリッド距離 | $\|\mathbf{a} - \mathbf{b}\|_2$ | 位置と大きさの両方を考慮 | クラスタリング、kNN |
| コサイン類似度 | $\frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$ | 方向のみ（大きさ無関係） | 文書検索、意味類似度 |
| ドット積 | $\mathbf{a} \cdot \mathbf{b}$ | 方向 + 大きさ | Attention、推薦 |

**正規化ベクトル（$\|\mathbf{v}\| = 1$）の場合の等価関係:**

$$\cos(\mathbf{a}, \mathbf{b}) = \mathbf{a} \cdot \mathbf{b}$$

$$\|\mathbf{a} - \mathbf{b}\|^2 = 2(1 - \cos(\mathbf{a}, \mathbf{b}))$$

### 8.3 よくある間違い

#### 間違い 1: コサイン類似度を使う前に正規化が必要だと思い込む

コサイン類似度の数式自体にノルムでの除算が含まれているため、**入力ベクトルを事前に正規化する必要はありません**。ただし、計算効率のために事前に正規化しておき、ドット積で代用するのは正当なテクニックです。

#### 間違い 2: 正規化されていない埋め込みにユークリッド距離をそのまま使う

ベクトルのノルムがばらばらの場合、ユークリッド距離はノルムの差に大きく影響されます。意味の方向ではなく、ベクトルの「大きさ」の違いをランキングに反映してしまうことがあります。正規化してから使うか、コサイン類似度を使いましょう。

#### 間違い 3: 高次元空間での距離計算の意味を過信する

Section 6 で見たように、高次元ではランダムなベクトル間の距離がほぼ均一になります。これは「高い類似度 = 本当に意味的に近い」とは限らないことを意味します。閾値の設定や、次元削減の検討が必要な場合があります。

### 8.4 自己評価クイズ

---

**Q1.** コサイン類似度とドット積の違いを一文で説明してください。

<details>
<summary>回答を表示</summary>

コサイン類似度はベクトルの **方向のみ** を比較し（ノルムで除算するため大きさの影響がない）、ドット積は **方向と大きさの両方** を反映する。正規化済みベクトルでは両者は等価。

</details>

---

**Q2.** `king - man + woman ≈ queen` が成立するための幾何学的条件は何ですか？

<details>
<summary>回答を表示</summary>

4つの単語ベクトルが **平行四辺形** を形成すること。具体的には、`king - man ≈ queen - woman`（つまり「性別」方向の差分ベクトルが男性ペアと女性ペアでほぼ等しい）。これは意味空間に「性別」や「王族」といった **線形な意味軸** が存在することを意味する。

</details>

---

**Q3.** 1000次元空間でランダムな単位ベクトルを大量に生成した場合、ペアワイズのコサイン類似度の分布はどうなりますか？

<details>
<summary>回答を表示</summary>

ほぼ全てのペアのコサイン類似度が **0 付近に集中** する（標準偏差が非常に小さくなる）。これは高次元空間ではランダムなベクトルがほぼ直交するためであり、「高次元の呪い」の一つの現れ。

</details>

---

**Q4.** 文書検索で TF-IDF ベクトルの類似度を計算したい場合、ユークリッド距離とコサイン類似度のどちらが適切ですか？理由も述べてください。

<details>
<summary>回答を表示</summary>

**コサイン類似度** が適切。TF-IDF ベクトルは文書の長さによってノルムが大きく異なるため、ユークリッド距離は文書の長さの違いに過度に影響される。コサイン類似度はノルムを正規化するので、文書の長さに関係なく **内容の方向（トピック）** だけで類似度を測れる。

</details>

### 8.5 次のステップ

次の **Notebook 151: Word2Vec** では、ここで使った GloVe のような単語埋め込みが **どのように学習されるか** を掘り下げます。

- Skip-gram モデルと CBOW モデルの仕組み
- Negative Sampling の理論
- 小さなコーパスで Word2Vec を学習する実装

---

## 参考文献

1. Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*. EMNLP.
2. Mikolov, T., et al. (2013). *Efficient Estimation of Word Representations in Vector Space*. arXiv:1301.3781.
3. Aggarwal, C. C., Hinneburg, A., & Keim, D. A. (2001). *On the Surprising Behavior of Distance Metrics in High Dimensional Space*. ICDT.
4. 斎藤康毅 (2018). 『ゼロから作るDeep Learning 2 ― 自然言語処理編』. O'Reilly Japan.